In [8]:
from godot_rl.core.godot_env import GodotEnv
from dataclasses import dataclass
from collections import deque

from torch.distributions.categorical import Categorical

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Create GodotEnv
env = GodotEnv(show_window=True)

# Get action space info
# GodotEnv returns list of action dicts per agent
# Each agent has: accelerate_forward (3), accelerate_sideways (3), turn (3), shoot (2)
print(f"Environment: GodotEnv")
print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")

Using device: mps
No game binary has been provided, please press PLAY in the Godot editor
waiting for remote GODOT connection on port 11008
connection established
action space [{'accelerate_forward': {'size': 3, 'action_type': 'discrete'}, 'accelerate_sideways': {'size': 3, 'action_type': 'discrete'}, 'turn': {'size': 3, 'action_type': 'discrete'}, 'shoot': {'size': 2, 'action_type': 'discrete'}}, {'accelerate_forward': {'size': 3, 'action_type': 'discrete'}, 'accelerate_sideways': {'size': 3, 'action_type': 'discrete'}, 'turn': {'size': 3, 'action_type': 'discrete'}, 'shoot': {'size': 2, 'action_type': 'discrete'}}, {'accelerate_forward': {'size': 3, 'action_type': 'discrete'}, 'accelerate_sideways': {'size': 3, 'action_type': 'discrete'}, 'turn': {'size': 3, 'action_type': 'discrete'}, 'shoot': {'size': 2, 'action_type': 'discrete'}}]
observation space [{'right_eye': {'size': [3, 300, 320], 'space': 'box'}, 'left_eye': {'size': [3, 300, 320], 'space': 'box'}, 'hp': {'size': [1], 'spa

In [9]:
# Test observation structure
obs, _ = env.reset()
print(f"Number of agents: {len(obs)}")
print(f"Agent 0 observation keys: {obs[0].keys()}")

# Decode and check image shapes
right_eye_hex = obs[0]['right_eye']
right_eye_bytes = bytes.fromhex(right_eye_hex)
right_eye_arr = np.frombuffer(right_eye_bytes, dtype=np.uint8)
print(f"Right eye raw data length: {len(right_eye_arr)}")

left_eye_hex = obs[0]['left_eye']
left_eye_bytes = bytes.fromhex(left_eye_hex)
left_eye_arr = np.frombuffer(left_eye_bytes, dtype=np.uint8)
print(f"Left eye raw data length: {len(left_eye_arr)}")

hp = obs[0]['hp']
print(f"HP: {hp}")

Number of agents: 3
Agent 0 observation keys: dict_keys(['right_eye', 'left_eye', 'hp'])
Right eye raw data length: 288000
Left eye raw data length: 288000
HP: [5.99953703703704]


In [10]:
# GodotEnv Preprocessor
class GodotPreprocessor:
    """Preprocesses GodotEnv observations.
    
    Combines:
    - left_eye (320 width, 300 height) and right_eye (320 width, 300 height) images
    - Previous output (out_next) from model
    - Current HP
    
    Process:
    - Stack left_eye and right_eye vertically: (3, 600, 320)
    - HP embedded as (3, 40, 320)
    - Combined observation: (3, 640, 320)
    - Concatenate with out_next (3, 640, 320) horizontally: (3, 640, 640)
    
    Final input shape: (3, 640, 640)
    """
    
    def __init__(self):
        self.original_eye_size = (300, 320)  # (height, width) from GodotEnv
        self.hp_height = 40  # Height for HP bar
        self.out_next_shape = (3, 640, 320)  # Shape of model output / previous output
        
    def reset(self):
        """No state to clear."""
        pass
    
    def decode_eye(self, eye_hex):
        """Decode hex string to numpy array (H, W, 3)."""
        eye_bytes = bytes.fromhex(eye_hex)
        arr = np.frombuffer(eye_bytes, dtype=np.uint8)
        # Original shape is (height=300, width=320, 3)
        img = arr.reshape(self.original_eye_size[0], self.original_eye_size[1], 3)
        return img
    
    def get_state(self, obs_dict):
        """Process observation dict from a single agent.
        
        Args:
            obs_dict: dict with 'left_eye', 'right_eye', 'hp' keys
            
        Returns:
            combined_eyes: (3, 600, 320) - vertically stacked eyes
            hp: float - health value normalized to [0, 1]
        """
        # Decode eyes - each is (300, 320, 3) = (height, width, channels)
        left_eye = self.decode_eye(obs_dict['left_eye'])  # (300, 320, 3)
        right_eye = self.decode_eye(obs_dict['right_eye'])  # (300, 320, 3)
        
        # Stack vertically without resizing: (600, 320, 3)
        combined = np.concatenate([left_eye, right_eye], axis=0)
        
        # Normalize to [0, 1] and transpose to (3, 600, 320)
        combined = combined.astype(np.float32) / 255.0
        combined = np.transpose(combined, (2, 0, 1))
        
        # Get HP (it's a list with one element, normalize assuming max HP is 100)
        hp_value = obs_dict['hp']
        if isinstance(hp_value, (list, np.ndarray)):
            hp_value = hp_value[0]  # Extract first element from list
        hp = float(hp_value) / 10.0  # Normalize HP
        
        return combined, hp
    
    def combine_state_with_out_next_and_hp(self, combined_eyes, hp, out_next=None):
        """Combine eyes, hp, and out_next into final input.
        
        Args:
            combined_eyes: (3, 600, 320) - vertically stacked eyes
            hp: float - normalized health value
            out_next: (3, 640, 320) or None - previous model output
            
        Returns:
            combined: (3, 640, 640) - full input for model
        """
        _, eye_h, eye_w = combined_eyes.shape  # (3, 600, 320)
        
        # Create HP embedding: broadcast hp value to (3, 40, 320)
        hp_embed = np.full((3, self.hp_height, eye_w), hp, dtype=np.float32)
        
        # Concatenate vertically: eyes (3, 600, 320) + hp (3, 40, 320) = (3, 640, 320)
        current_obs = np.concatenate([combined_eyes, hp_embed], axis=1)
        
        # If out_next is None, create zeros (3, 640, 320)
        if out_next is None:
            out_next = np.zeros(self.out_next_shape, dtype=np.float32)
        
        # Concatenate horizontally: current_obs (3, 640, 320) + out_next (3, 640, 320) = (3, 640, 640)
        combined = np.concatenate([current_obs, out_next], axis=2)
        
        return combined
    
    def get_input_shape(self):
        """Return the expected input shape for the model."""
        return (3, 640, 640)
    
    def get_out_next_shape(self):
        """Return the expected shape of out_next (model output)."""
        return self.out_next_shape  # (3, 640, 320)


# Test the preprocessor
preprocessor = GodotPreprocessor()
combined_eyes, hp = preprocessor.get_state(obs[0])
print(f"Combined eyes shape: {combined_eyes.shape}")  # Should be (3, 600, 320)
print(f"HP value: {hp}")

combined_full = preprocessor.combine_state_with_out_next_and_hp(combined_eyes, hp)
print(f"Full combined shape (with zeros out_next): {combined_full.shape}")  # Should be (3, 640, 640)
print(f"Expected input shape: {preprocessor.get_input_shape()}")  # (3, 640, 640)
print(f"Expected out_next shape: {preprocessor.get_out_next_shape()}")  # (3, 640, 320)

Combined eyes shape: (3, 600, 320)
HP value: 0.599953703703704
Full combined shape (with zeros out_next): (3, 640, 640)
Expected input shape: (3, 640, 640)
Expected out_next shape: (3, 640, 320)


In [11]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Action dimensions for GodotEnv
ACTION_DIMS = {
    'accelerate_forward': 3,
    'accelerate_sideways': 3,
    'turn': 3,
    'shoot': 2
}
TOTAL_ACTIONS = sum(ACTION_DIMS.values())  # 3 + 3 + 3 + 2 = 11


class MobileNetV3PolicyGodot(nn.Module):
    """MobileNetV3-based policy for GodotEnv with multi-discrete actions.
    
    Input: (3, 640, 640) - combined (eyes + hp) + out_next
    Output: (3, 640, 320) - used for action selection and as out_next for next step
    
    Action space: Multi-discrete
    - accelerate_forward: 3 choices
    - accelerate_sideways: 3 choices  
    - turn: 3 choices
    - shoot: 2 choices
    """
    
    def __init__(self, pretrained=True, freeze_backbone=False):
        super(MobileNetV3PolicyGodot, self).__init__()
        self.action_dims = ACTION_DIMS
        self.total_actions = TOTAL_ACTIONS
        self.out_h = 640  # Output height
        self.out_w = 320  # Output width
        
        # Load pretrained MobileNetV3-Small
        if pretrained:
            weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
            backbone = mobilenet_v3_small(weights=weights)
        else:
            backbone = mobilenet_v3_small(weights=None)
        
        # Keep only the feature extractor
        # Input: (batch, 3, 640, 640)
        # MobileNetV3-Small features output: (batch, 576, H/32, W/32)
        # For 640x640 input: output is (batch, 576, 20, 20)
        self.features = backbone.features
        
        if freeze_backbone:
            for param in self.features.parameters():
                param.requires_grad = False
        
        # Upsample to output shape (3, 640, 320)
        self.upsample_conv1 = nn.Sequential(
            nn.Conv2d(576, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.upsample_conv2 = nn.Sequential(
            nn.Conv2d(256, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.upsample_conv3 = nn.Conv2d(64, 3, kernel_size=3, padding=1)
        
        self._init_upsample_weights()
    
    def _init_upsample_weights(self):
        for m in [self.upsample_conv1, self.upsample_conv2, self.upsample_conv3]:
            for layer in m.modules() if isinstance(m, nn.Sequential) else [m]:
                if isinstance(layer, nn.Conv2d):
                    nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        """Forward pass, returns (batch, 3, 640, 320)."""
        # Extract features: (batch, 3, 640, 640) -> (batch, 576, 20, 20)
        features = self.features(x)
        
        # Upsample using interpolate for flexible sizing
        x = self.upsample_conv1(features)  # (batch, 256, 20, 20)
        x = F.interpolate(x, size=(160, 80), mode='bilinear', align_corners=False)
        
        x = self.upsample_conv2(x)  # (batch, 64, 160, 80)
        x = F.interpolate(x, size=(self.out_h, self.out_w), mode='bilinear', align_corners=False)
        
        # Final conv: (batch, 64, 640, 320) -> (batch, 3, 640, 320)
        out = self.upsample_conv3(x)
        return out
    
    def extract_action_logits_from_crops(self, out):
        """Extract logits for each action choice from 11 consecutive 3x8x8 crops.
        
        Args:
            out: tensor (batch, 3, 640, 320) - model output
            
        Returns:
            logits: list of tensors, one per action type with shape (batch, action_size)
        """
        # Extract 11 crops: each crop is 3x2x2
        crop_values = []
        for i in range(self.total_actions):
            crop = out[:, :, 16:17, (16 + i*2):(16 + (i+1)*2)]  # (batch, 3, 2, 2)
            crop_mean = crop.mean(dim=(1, 2, 3))  # (batch,) - mean across channels and spatial dims
            crop_values.append(crop_mean)
        
        # Stack all crop values: (batch, 11)
        all_logits = torch.stack(crop_values, dim=1)  # (batch, 11)
        
        # Split into action-specific logits
        logits = []
        idx = 0
        for action_name, action_size in self.action_dims.items():
            action_logits = all_logits[:, idx:idx + action_size]  # (batch, action_size)
            logits.append(action_logits)
            idx += action_size
        
        return logits
    
    def act(self, combined_state):
        """Select actions and generate out_next for next step.
        
        Args:
            combined_state: (3, 640, 640) - combined (eyes + hp) + out_next
            
        Returns:
            actions: dict with action keys
            log_prob: tensor - sum of log probabilities
            out_next: numpy array (3, 640, 320) - output for next step's input
        """
        if len(combined_state.shape) == 3:
            combined_state = combined_state[np.newaxis, ...]
        
        state_tensor = torch.FloatTensor(combined_state).to(device)
        out = self.forward(state_tensor)  # (1, 3, 640, 320)
        
        # Extract action logits from first 11 3x8x8 crops
        action_logits_list = self.extract_action_logits_from_crops(out)
        
        # Select actions for each action dimension
        actions = {}
        log_probs = []
        
        for (action_name, action_size), action_logits in zip(self.action_dims.items(), action_logits_list):
            probs = F.softmax(action_logits, dim=1).cpu()
            
            m = Categorical(probs)
            action = m.sample()
            log_prob = m.log_prob(action)
            
            actions[action_name] = action.item()
            log_probs.append(log_prob)
        
        # Sum log probs for total action probability
        total_log_prob = torch.stack(log_probs).sum()
        
        # out_next is directly the model output (3, 640, 320)
        out_next = out.squeeze(0).detach().cpu().numpy()  # (3, 640, 320)
        
        return actions, total_log_prob, out_next


# Create and test the model
policy = MobileNetV3PolicyGodot(pretrained=True, freeze_backbone=False).to(device)
print(f"MobileNetV3 Policy for GodotEnv:")
print(policy)

# Count parameters
total_params = sum(p.numel() for p in policy.parameters())
trainable_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass with correct input size
test_input = torch.randn(1, 3, 640, 640).to(device)
test_output = policy(test_input)
print(f"\nTest input shape: {test_input.shape}")  # (1, 3, 640, 640)
print(f"Test output shape: {test_output.shape}")  # (1, 3, 640, 320)

MobileNetV3 Policy for GodotEnv:
MobileNetV3PolicyGodot(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
      

In [12]:
# Test the act function
combined_eyes, hp = preprocessor.get_state(obs[0])
combined_full = preprocessor.combine_state_with_out_next_and_hp(combined_eyes, hp)
print(f"Combined eyes shape: {combined_eyes.shape}")  # Should be (3, 600, 320)
print(f"Input shape: {combined_full.shape}")  # Should be (3, 640, 640)

actions, log_prob, out_next = policy.act(combined_full)
print(f"Actions: {actions}")
print(f"Log prob: {log_prob}")
print(f"Out next shape: {out_next.shape}")  # Should be (3, 640, 320)

# Test with out_next feedback
combined_full_with_out_next = preprocessor.combine_state_with_out_next_and_hp(combined_eyes, hp, out_next)
print(f"Input with out_next shape: {combined_full_with_out_next.shape}")  # Should be (3, 640, 640)

Combined eyes shape: (3, 600, 320)
Input shape: (3, 640, 640)
Actions: {'accelerate_forward': 0, 'accelerate_sideways': 1, 'turn': 1, 'shoot': 1}
Log prob: -4.003784656524658
Out next shape: (3, 640, 320)
Input with out_next shape: (3, 640, 640)


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def extract_loss_and_lr_regions(out_next):
    """
    Extract the middle 3x8x8 region (LOSS region) and adjacent region (LR region) from model output.
    
    Model output shape: (3, 640, 320)
    LOSS region: center 3x8x8 - used for computing loss
    LR region: adjacent 3x8x8 to the right - used for determining learning rate
    
    Args:
        out_next: numpy array (3, 640, 320)
        
    Returns:
        loss_region: (3, 2, 2) - center region for loss
        lr_region: (3, 2, 2) - adjacent region for learning rate
    """
    h_center = 320
    w_center = 160
    half_size = 1
    
    loss_region = out_next[:, h_center-half_size:h_center+half_size, w_center-half_size:w_center+half_size]
    lr_region = out_next[:, h_center-half_size:h_center+half_size, w_center+half_size:w_center+half_size*2]
    
    return loss_region, lr_region


def compute_loss_from_region(loss_region):
    """
    Compute loss value from the LOSS region.
    """
    mean_abs = torch.abs(loss_region).mean()
    variance = loss_region.var()
    loss = mean_abs - 0.1 * variance
    return loss


def compute_lr_from_region(lr_region, lr_min=1e-6, lr_max=1e-4):
    """
    Compute learning rate from the LR region normalized.
    """
    lr_mean = np.mean(lr_region)
    normalized = (np.tanh(lr_mean) + 1) / 2
    lr = lr_min + normalized * (lr_max - lr_min)
    return float(lr)


def reinforce_godot_multi_agent(policies, optimizers, preprocessor, env,
                                n_training_episodes, max_t, gamma, print_every,
                                lr_min=1e-6, lr_max=1e-4, death_lr=1e-2, n_agents=2,
                                evolve_every=1000, mutation_strength=0.01):
    """
    多智能体进化式训练 - 每个智能体拥有独立的模型参数。
    
    核心设计（类进化算法）:
    1. 每个智能体拥有独立的策略网络（不同的模型参数）和独立的优化器
    2. 每步每个智能体独立处理自己的观测、计算损失、更新参数
    3. 死亡惩罚仅施加于死亡的智能体
    4. 进化机制: 定期将最佳智能体的参数（加高斯变异）复制给最差智能体
    
    Args:
        policies: list[MobileNetV3PolicyGodot] - 每个智能体的独立策略网络
        optimizers: list[Optimizer] - 每个智能体的独立优化器
        preprocessor: GodotPreprocessor 实例
        env: GodotEnv 环境
        n_agents: 环境中的智能体数量
        evolve_every: 每隔多少步执行一次进化选择
        mutation_strength: 进化变异的高斯噪声标准差
    """
    # === 每个智能体的独立状态追踪 ===
    agent_out_nexts = [None] * n_agents          # 每个智能体的 out_next
    agent_survival_steps = [0] * n_agents        # 当前生命的存活步数
    agent_total_survival = [0] * n_agents        # 累计存活步数（适应度指标）
    agent_deaths = [0] * n_agents                # 累计死亡次数
    agent_state_tensors = [None] * n_agents      # 保存 state_tensor 用于死亡惩罚
    
    scores = []
    best_score = -np.inf
    
    for i_episode in range(1, n_training_episodes + 1):
        obs, _ = env.reset()
        preprocessor.reset()
        
        # 初始化每个智能体的观测状态
        agent_eyes = []
        agent_hps = []
        for agent_idx in range(n_agents):
            eyes, hp = preprocessor.get_state(obs[agent_idx])
            agent_eyes.append(eyes)
            agent_hps.append(hp)
        
        agent_out_nexts = [None] * n_agents
        agent_survival_steps = [0] * n_agents
        
        episode_reward = 0
        total_steps = 0
        
        for t in range(max_t):
            all_actions = []
            
            # =========================================================
            # 每个智能体独立执行: 观测 → 前向传播 → 损失 → 更新 → 动作选择
            # =========================================================
            for agent_idx in range(n_agents):
                policy = policies[agent_idx]
                optimizer = optimizers[agent_idx]
                
                # 组合当前观测 + 上一步输出 + HP
                combined_state = preprocessor.combine_state_with_out_next_and_hp(
                    agent_eyes[agent_idx], agent_hps[agent_idx], agent_out_nexts[agent_idx]
                )
                
                if len(combined_state.shape) == 3:
                    state_batch = combined_state[np.newaxis, ...]
                else:
                    state_batch = combined_state
                state_tensor = torch.FloatTensor(state_batch).to(device)
                agent_state_tensors[agent_idx] = state_tensor  # 保存用于死亡惩罚
                
                # 前向传播（需要梯度用于损失计算）
                model_output = policy(state_tensor)  # (1, 3, 640, 320)
                out_numpy = model_output.squeeze(0).detach().cpu().numpy()
                
                # 提取 LOSS 和 LR 区域
                loss_region_np, lr_region_np = extract_loss_and_lr_regions(out_numpy)
                dynamic_lr = compute_lr_from_region(lr_region_np, lr_min, lr_max)
                
                # 更新此智能体的学习率
                for param_group in optimizer.param_groups:
                    param_group['lr'] = dynamic_lr
                
                # 计算损失并更新此智能体的参数
                loss_region_tensor = model_output[:, :, 316:324, 156:164]
                loss = compute_loss_from_region(loss_region_tensor.squeeze(0))
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
                optimizer.step()
                
                # 从模型输出中提取动作（使用 detached output）
                action_logits_list = policy.extract_action_logits_from_crops(model_output.detach())
                actions = {}
                for (action_name, action_size), action_logits in zip(policy.action_dims.items(), action_logits_list):
                    probs = F.softmax(action_logits, dim=1).cpu()
                    m = Categorical(probs)
                    action = m.sample()
                    actions[action_name] = action.item()
                
                # 保存 out_next 用于下一步输入
                agent_out_nexts[agent_idx] = out_numpy
                
                # 格式化动作
                agent_action = [
                    actions['accelerate_forward'],
                    actions['accelerate_sideways'],
                    actions['turn'],
                    actions['shoot']
                ]
                all_actions.append(agent_action)
            
            # =========================================================
            # 环境执行一步（所有智能体同时行动）
            # =========================================================
            obs, reward_list, terminated_list, truncated_list, _ = env.step(all_actions, True)
            total_steps += 1
            
            # =========================================================
            # 处理每个智能体的反馈
            # =========================================================
            dead_agents = []
            for agent_idx in range(n_agents):
                reward = reward_list[agent_idx] if isinstance(reward_list, list) else reward_list
                episode_reward += reward
                agent_survival_steps[agent_idx] += 1
                agent_total_survival[agent_idx] += 1
                
                # 获取下一状态
                agent_eyes[agent_idx], agent_hps[agent_idx] = preprocessor.get_state(obs[agent_idx])
                current_hp = agent_hps[agent_idx] * 10
                
                # 检测死亡
                if current_hp <= 0:
                    dead_agents.append(agent_idx)
            
            # =========================================================
            # 处理死亡的智能体 - 仅对死亡智能体施加死亡惩罚
            # =========================================================
            if dead_agents:
                for agent_idx in dead_agents:
                    agent_deaths[agent_idx] += 1
                    survived = agent_survival_steps[agent_idx]
                    print(f"  [Step {total_steps}] 智能体 {agent_idx} 死亡! "
                          f"存活 {survived} 步 (累计死亡: {agent_deaths[agent_idx]})")
                    
                    # 对死亡智能体施加死亡惩罚（反向梯度 + 大学习率）
                    policy = policies[agent_idx]
                    optimizer = optimizers[agent_idx]
                    state_tensor = agent_state_tensors[agent_idx]
                    
                    model_output_death = policy(state_tensor)
                    loss_region_death = model_output_death[:, :, 316:324, 156:164]
                    loss_death = compute_loss_from_region(loss_region_death.squeeze(0))
                    loss2 = -loss_death  # 反转梯度方向
                    
                    optimizer.zero_grad()
                    loss2.backward()
                    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
                    
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = death_lr
                    optimizer.step()
                    
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = lr_max
                    
                    print(f"    已施加死亡惩罚 (death_lr={death_lr:.0e})")

                    agent_eyes[agent_idx], agent_hps[agent_idx] = preprocessor.get_state(obs[agent_idx])
                    agent_out_nexts[agent_idx] = None
                    agent_survival_steps[agent_idx] = 0
                
                # 有智能体死亡 → 重置环境（影响所有智能体）
                obs, _ = env.reset()
                preprocessor.reset()
                # for agent_idx in range(n_agents):
                #     agent_eyes[agent_idx], agent_hps[agent_idx] = preprocessor.get_state(obs[agent_idx])
                #     agent_out_nexts[agent_idx] = None
                #     agent_survival_steps[agent_idx] = 0
                continue
            
            
            # =========================================================
            # 进化选择机制 - 定期用最佳智能体替换最差智能体
            # =========================================================
            # if total_steps % evolve_every == 0 and n_agents > 1:
            #     best_idx = int(np.argmax(agent_total_survival))
            #     worst_idx = int(np.argmin(agent_total_survival))
                
            #     if best_idx != worst_idx:
            #         print(f"\n  {'='*50}")
            #         print(f"  进化选择 @ Step {total_steps}")
            #         print(f"  {'='*50}")
            #         for i in range(n_agents):
            #             marker = " ★最佳" if i == best_idx else (" ✗最差" if i == worst_idx else "")
            #             print(f"    智能体 {i}: 累计存活={agent_total_survival[i]}, "
            #                   f"死亡次数={agent_deaths[i]}{marker}")
                    
            #         # 将最佳智能体的参数（加高斯变异）复制给最差智能体
            #         best_state = policies[best_idx].state_dict()
            #         new_state = {}
            #         for key in best_state:
            #             new_state[key] = best_state[key].clone() + \
            #                 torch.randn_like(best_state[key]) * mutation_strength
                    
            #         policies[worst_idx].load_state_dict(new_state)
            #         # 重建最差智能体的优化器（清除旧的动量状态）
            #         optimizers[worst_idx] = optim.Adam(
            #             policies[worst_idx].parameters(), lr=lr_max
            #         )
                    
            #         # 重置最差智能体的适应度，给它公平的评估机会
            #         agent_total_survival[worst_idx] = agent_total_survival[best_idx]
            #         agent_deaths[worst_idx] = agent_deaths[best_idx]
                    
            #         print(f"  → 智能体 {best_idx} 的参数(+变异) 复制到 智能体 {worst_idx}")
            #         print(f"    变异强度: {mutation_strength}")
            #         print(f"  {'='*50}\n")
            
            # =========================================================
            # 定期打印所有智能体的状态
            # =========================================================
            if total_steps % 100 == 0:
                status_parts = []
                for i in range(n_agents):
                    status_parts.append(
                        f"A{i}[HP={agent_hps[i]*10:.2f}, "
                        f"存活={agent_survival_steps[i]}, "
                        f"死亡={agent_deaths[i]}]"
                    )
                status = " | ".join(status_parts)
                print(f"  [Step {total_steps}] {status}")
        
        # =========================================================
        # Episode 结束 - 统计和保存
        # =========================================================
        scores.append(episode_reward)
        
        if episode_reward > best_score:
            best_score = episode_reward
        
        # 保存每个智能体的模型
        for agent_idx in range(n_agents):
            torch.save(policies[agent_idx].state_dict(),
                       f"saved_models/agent_{agent_idx}.pth")
        
        # 保存最佳智能体（累计存活最高）
        best_agent = int(np.argmax(agent_total_survival))
        torch.save(policies[best_agent].state_dict(), "saved_models/best_agent.pth")
        
        print(f"\n{'='*60}")
        print(f"Episode {i_episode} 完成")
        print(f"{'='*60}")
        print(f"  总步数: {total_steps}")
        print(f"  总奖励: {episode_reward:.2f}")
        for i in range(n_agents):
            print(f"  智能体 {i}: 累计存活={agent_total_survival[i]}, "
                  f"死亡次数={agent_deaths[i]}")
        print(f"  最佳智能体: {best_agent} (累计存活={agent_total_survival[best_agent]})")
        print(f"{'='*60}\n")
    
    return scores

In [ ]:
# =================================================================
# 多智能体进化式训练 - 超参数配置
# =================================================================
# 每个智能体拥有独立的 MobileNetV3 策略网络（不同参数）
# 通过进化选择机制，表现好的智能体参数（加变异）替换表现差的
# =================================================================

hyperparameters = {
    "n_training_episodes": 1,
    "n_evaluation_episodes": 1,
    "max_t": 1000000,       # 单个 episode 最大步数
    "gamma": 0.99,
    "lr_min": 1e-5,         # 最小学习率
    "lr_max": 2e-4,         # 最大学习率
    "death_lr": 1e-3,       # 死亡惩罚时的大学习率
    "pretrained": True,
    "freeze_backbone": False,
    "n_agents": 3,           # 环境中的智能体数量（需与 Godot 场景中的数量一致）
    "evolve_every": 1000,    # 每隔多少步执行一次进化选择
    "mutation_strength": 0.01,  # 进化变异的高斯噪声标准差
}

n_agents = hyperparameters["n_agents"]

# =================================================================
# 为每个智能体创建独立的策略网络和优化器
# 注意: 虽然 backbone (MobileNetV3) 初始权重相同（pretrained），
#       但 upsample 层是随机初始化的，所以每个智能体的参数天然不同。
#       训练过程中它们会因为不同的经历而进一步分化。
# =================================================================
godot_policies = []
godot_optimizers = []

for i in range(n_agents):
    policy = MobileNetV3PolicyGodot(
        pretrained=hyperparameters["pretrained"],
        freeze_backbone=hyperparameters["freeze_backbone"],
    ).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=hyperparameters["lr_max"])
    godot_policies.append(policy)
    godot_optimizers.append(optimizer)
    
    trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    print(f"  智能体 {i}: 已创建独立策略网络 (可训练参数: {trainable:,})")

godot_preprocessor = GodotPreprocessor()

print(f"\n{'='*60}")
print(f"多智能体进化式训练配置")
print(f"{'='*60}")
print(f"  智能体数量: {n_agents}")
print(f"  设备: {device}")
print(f"  Pretrained backbone: {hyperparameters['pretrained']}")
print(f"  动态学习率范围: [{hyperparameters['lr_min']:.0e}, {hyperparameters['lr_max']:.0e}]")
print(f"  死亡惩罚学习率: {hyperparameters['death_lr']:.0e}")
print(f"  进化选择间隔: 每 {hyperparameters['evolve_every']} 步")
print(f"  变异强度: {hyperparameters['mutation_strength']}")
print(f"{'='*60}\n")

# =================================================================
# 开始多智能体进化式训练
# =================================================================
scores = reinforce_godot_multi_agent(
    godot_policies,
    godot_optimizers,
    godot_preprocessor,
    env,
    hyperparameters["n_training_episodes"],
    hyperparameters["max_t"],
    hyperparameters["gamma"],
    print_every=10,
    lr_min=hyperparameters["lr_min"],
    lr_max=hyperparameters["lr_max"],
    death_lr=hyperparameters["death_lr"],
    n_agents=hyperparameters["n_agents"],
    evolve_every=hyperparameters["evolve_every"],
    mutation_strength=hyperparameters["mutation_strength"],
)

# 保存最终所有模型
for i in range(n_agents):
    torch.save(godot_policies[i].state_dict(), f"saved_models/agent_{i}_final.pth")
    print(f"智能体 {i} 最终模型已保存: saved_models/agent_{i}_final.pth")

print("\n训练完成! 所有模型已保存。")

  智能体 0: 已创建独立策略网络 (可训练参数: 2,403,619)
  智能体 1: 已创建独立策略网络 (可训练参数: 2,403,619)
  智能体 2: 已创建独立策略网络 (可训练参数: 2,403,619)

多智能体进化式训练配置
  智能体数量: 3
  设备: mps
  Pretrained backbone: True
  动态学习率范围: [1e-05, 2e-04]
  死亡惩罚学习率: 1e-03
  进化选择间隔: 每 1000 步
  变异强度: 0.01

  [Step 100] A0[HP=57, 存活=100, 死亡=0] | A1[HP=7, 存活=100, 死亡=0] | A2[HP=107, 存活=100, 死亡=0]
  [Step 200] A0[HP=3, 存活=200, 死亡=0] | A1[HP=3, 存活=200, 死亡=0] | A2[HP=93, 存活=200, 死亡=0]
  [Step 283] 智能体 0 死亡! 存活 283 步 (累计死亡: 1)
    已施加死亡惩罚 (death_lr=1e-03)
  [Step 284] 智能体 1 死亡! 存活 284 步 (累计死亡: 1)
    已施加死亡惩罚 (death_lr=1e-03)
  [Step 300] A0[HP=10, 存活=17, 死亡=1] | A1[HP=10, 存活=16, 死亡=1] | A2[HP=49, 存活=300, 死亡=0]
  [Step 394] 智能体 1 死亡! 存活 110 步 (累计死亡: 2)
    已施加死亡惩罚 (death_lr=1e-03)
  [Step 400] A0[HP=6, 存活=117, 死亡=1] | A1[HP=110, 存活=6, 死亡=2] | A2[HP=46, 存活=400, 死亡=0]
  [Step 426] 智能体 0 死亡! 存活 143 步 (累计死亡: 2)
    已施加死亡惩罚 (death_lr=1e-03)
  [Step 427] 智能体 0 死亡! 存活 1 步 (累计死亡: 3)
    已施加死亡惩罚 (death_lr=1e-03)
  [Step 500] A0[HP=8, 存活=73, 死亡=3] | A1[HP=57

In [ ]:
# Close environment when done
env.close()
print("Environment closed.")

close message sent
Environment closed.
